# Notebook 5 — Feature Engineering

Building the actual features here, based on what came out of notebook 4.

**Reads**: artifacts/train.parquet, artifacts/val.parquet, artifacts/test.parquet
**Writes**: artifacts/train_features.parquet, val_features.parquet, test_features.parquet,
artifacts/preprocessor.pkl, artifacts/top_states.pkl, artifacts/top_seller_states.pkl,
artifacts/feature_list.txt

In [23]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)

ARTIFACTS_DIR = Path("artifacts")

train = pd.read_parquet(ARTIFACTS_DIR / "train.parquet")
val = pd.read_parquet(ARTIFACTS_DIR / "val.parquet")
test = pd.read_parquet(ARTIFACTS_DIR / "test.parquet")

train.shape, val.shape, test.shape

((67533, 33), (14471, 33), (14472, 33))

## First, dropping anything that isn't actually known at prediction time

This is the part worth being careful about. The model needs to predict
lateness at the moment an order is placed, before it's shipped. A few
columns in this table only exist because the order already got delivered,
using them would mean training on information that literally can't exist
yet at prediction time.

order_delivered_customer_date and order_delivered_carrier_date describe
things that happen during fulfillment, not known at order time.
delivery_gap_days is calculated directly from the delivery date, it's
basically a disguised version of the label, has to go. avg_review_score and
num_reviews only exist after a customer received the order and left a
review, so even though notebook 4 flagged avg_review_score as correlated
with is_late, it can't be used, a review can't exist before the delivery
it's reviewing. order_approved_at is borderline (happens shortly after
purchase) but dropping it too, being conservative about what "prediction
time" actually means here. order_status is also gone since notebook 4 found
it's constant (all delivered) after the notebook 2 filtering, no
information in a column with one value.

order_estimated_delivery_date stays for now, Olist tells the customer this
promise at the time of purchase, so it's fair game, just not in raw date
form, turning it into a day count below.

In [24]:
LEAKY_OR_UNUSABLE = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_approved_at",
    "delivery_gap_days",
    "avg_review_score",
    "num_reviews",
    "order_status",
]

NOT_MODEL_FEATURES = [
    "order_id", "customer_id", "customer_unique_id",
    "customer_city", "customer_zip_code_prefix",
]

def drop_unusable(df):
    return df.drop(columns=[c for c in LEAKY_OR_UNUSABLE if c in df.columns])

train = drop_unusable(train)
val = drop_unusable(val)
test = drop_unusable(test)

train.columns.tolist()

['order_id',
 'customer_id',
 'order_purchase_timestamp',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'num_items',
 'num_distinct_products',
 'num_distinct_sellers',
 'total_price',
 'total_freight_value',
 'avg_item_price',
 'num_payments',
 'total_payment_value',
 'max_installments',
 'main_payment_type',
 'avg_distance_km',
 'total_product_weight_g',
 'avg_product_weight_g',
 'avg_product_length_cm',
 'avg_product_height_cm',
 'avg_product_width_cm',
 'main_seller_state',
 'is_late']

## Building the date-based features

Turning the two remaining date columns into numbers a model can use, then
dropping the raw timestamps themselves.

In [25]:
def engineer_dates(df):
    df = df.copy()
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_weekday"] = df["order_purchase_timestamp"].dt.dayofweek
    # months that stood out in notebook 3/4 as having a noticeably higher late rate
    df["high_risk_month"] = df["purchase_month"].isin([11, 2, 3]).astype(int)
    return df.drop(columns=["order_purchase_timestamp", "order_estimated_delivery_date"])

train = engineer_dates(train)
val = engineer_dates(val)
test = engineer_dates(test)

train[["estimated_delivery_days", "purchase_month", "purchase_weekday", "high_risk_month"]].head()

,estimated_delivery_days,purchase_month,purchase_weekday,high_risk_month
0,18.488449,9,3,0
1,23.593866,10,0,0
2,34.293866,10,0,0
3,52.123831,10,0,0
4,56.115556,10,0,0


## Bucketing the rare customer states

Notebook 4 found customer_state is dominated by a handful of states with a
long tail. Deciding which states count as "common enough to keep" using
train only, then applying that same decision to val and test, a state that's
common in train might be rare in val by chance and that's fine, the rule
itself came from train.

In [26]:
state_counts = train["customer_state"].value_counts()
top_states = state_counts[state_counts >= 500].index.tolist()
print(f"keeping {len(top_states)} states as their own category:", top_states)

def bucket_states(df, allowed_states):
    df = df.copy()
    df["customer_state_grouped"] = df["customer_state"].where(
        df["customer_state"].isin(allowed_states), "other"
    )
    return df.drop(columns=["customer_state"])

train = bucket_states(train, top_states)
val = bucket_states(val, top_states)
test = bucket_states(test, top_states)

train["customer_state_grouped"].value_counts()

keeping 16 states as their own category: ['SP', 'RJ', 'MG', 'RS', 'PR', 'SC', 'BA', 'ES', 'GO', 'DF', 'PE', 'CE', 'PA', 'MT', 'MA', 'MS']


customer_state_grouped
SP       27204
RJ        8967
MG        8130
RS        3912
PR        3461
SC        2577
BA        2307
other     2244
ES        1445
GO        1409
DF        1404
PE        1117
CE         948
PA         718
MT         640
MA         548
MS         502
Name: count, dtype: int64

In [27]:
# top_states is a decision made from train data, saving it so the exact
# same bucketing rule gets used later, on new data, without recomputing it
joblib.dump(top_states, ARTIFACTS_DIR / "top_states.pkl")

['artifacts/top_states.pkl']

## Bucketing rare seller states too

Added main_seller_state back in notebook 1 along with distance and product
weight, once notebook 6's first attempt showed the earlier features weren't
carrying much signal. Same long tail problem as customer_state, so bucketing
it the same way, rule learned from train.

In [28]:
seller_state_counts = train["main_seller_state"].value_counts()
top_seller_states = seller_state_counts[seller_state_counts >= 500].index.tolist()
print(f"keeping {len(top_seller_states)} seller states:", top_seller_states)

def bucket_seller_states(df, allowed_states):
    df = df.copy()
    df["main_seller_state_grouped"] = df["main_seller_state"].where(
        df["main_seller_state"].isin(allowed_states), "other"
    )
    return df.drop(columns=["main_seller_state"])

train = bucket_seller_states(train, top_seller_states)
val = bucket_seller_states(val, top_seller_states)
test = bucket_seller_states(test, top_seller_states)

joblib.dump(top_seller_states, ARTIFACTS_DIR / "top_seller_states.pkl")

train["main_seller_state_grouped"].value_counts()

keeping 7 seller states: ['SP', 'MG', 'PR', 'RJ', 'SC', 'RS', 'DF']


main_seller_state_grouped
SP       47773
MG        5664
PR        5385
RJ        2705
SC        2588
other     1544
RS        1287
DF         587
Name: count, dtype: int64

## Setting up the numeric and categorical pipelines

Missing values still need handling. Median imputation covers the numeric
columns, including the new distance and weight features, some rows are
missing those because a zip prefix didn't show up in the geolocation lookup,
median is a reasonable stand-in rather than dropping those rows. Most
frequent covers the two categorical columns in case an order somehow has
none. No scaling here, planning to use tree based models (random forest /
gradient boosting) starting in notebook 6, which don't care about feature
scale, if that changes later a scaler can get added to the numeric pipeline
without touching anything else.

In [29]:
numeric_features = [
    "num_items", "num_distinct_products", "num_distinct_sellers",
    "total_price", "total_freight_value", "avg_item_price",
    "num_payments", "total_payment_value", "max_installments",
    "estimated_delivery_days", "purchase_month", "purchase_weekday", "high_risk_month",
    "avg_distance_km", "total_product_weight_g", "avg_product_weight_g",
    "avg_product_length_cm", "avg_product_height_cm", "avg_product_width_cm",
]

categorical_features = ["customer_state_grouped", "main_seller_state_grouped", "main_payment_type"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

## Fitting on train only, applying to all three splits



In [30]:
X_train = preprocessor.fit_transform(train[numeric_features + categorical_features])
X_val = preprocessor.transform(val[numeric_features + categorical_features])
X_test = preprocessor.transform(test[numeric_features + categorical_features])

X_train.shape, X_val.shape, X_test.shape

((67533, 48), (14471, 48), (14472, 48))

In [31]:
feature_names = numeric_features + list(
    preprocessor.named_transformers_["cat"]["onehot"].get_feature_names_out(categorical_features)
)
len(feature_names), feature_names

(48,
 ['num_items',
  'num_distinct_products',
  'num_distinct_sellers',
  'total_price',
  'total_freight_value',
  'avg_item_price',
  'num_payments',
  'total_payment_value',
  'max_installments',
  'estimated_delivery_days',
  'purchase_month',
  'purchase_weekday',
  'high_risk_month',
  'avg_distance_km',
  'total_product_weight_g',
  'avg_product_weight_g',
  'avg_product_length_cm',
  'avg_product_height_cm',
  'avg_product_width_cm',
  'customer_state_grouped_BA',
  'customer_state_grouped_CE',
  'customer_state_grouped_DF',
  'customer_state_grouped_ES',
  'customer_state_grouped_GO',
  'customer_state_grouped_MA',
  'customer_state_grouped_MG',
  'customer_state_grouped_MS',
  'customer_state_grouped_MT',
  'customer_state_grouped_PA',
  'customer_state_grouped_PE',
  'customer_state_grouped_PR',
  'customer_state_grouped_RJ',
  'customer_state_grouped_RS',
  'customer_state_grouped_SC',
  'customer_state_grouped_SP',
  'customer_state_grouped_other',
  'main_seller_state_gr

In [32]:
train_features = pd.DataFrame(X_train, columns=feature_names)
train_features["order_id"] = train["order_id"].values
train_features["is_late"] = train["is_late"].values

val_features = pd.DataFrame(X_val, columns=feature_names)
val_features["order_id"] = val["order_id"].values
val_features["is_late"] = val["is_late"].values

test_features = pd.DataFrame(X_test, columns=feature_names)
test_features["order_id"] = test["order_id"].values
test_features["is_late"] = test["is_late"].values

train_features.head()

,num_items,num_distinct_products,num_distinct_sellers,total_price,total_freight_value,avg_item_price,num_payments,total_payment_value,max_installments,estimated_delivery_days,purchase_month,purchase_weekday,high_risk_month,avg_distance_km,total_product_weight_g,avg_product_weight_g,avg_product_length_cm,avg_product_height_cm,avg_product_width_cm,customer_state_grouped_BA,customer_state_grouped_CE,customer_state_grouped_DF,customer_state_grouped_ES,customer_state_grouped_GO,customer_state_grouped_MA,customer_state_grouped_MG,customer_state_grouped_MS,customer_state_grouped_MT,customer_state_grouped_PA,customer_state_grouped_PE,customer_state_grouped_PR,customer_state_grouped_RJ,customer_state_grouped_RS,customer_state_grouped_SC,customer_state_grouped_SP,customer_state_grouped_other,main_seller_state_grouped_DF,main_seller_state_grouped_MG,main_seller_state_grouped_PR,main_seller_state_grouped_RJ,main_seller_state_grouped_RS,main_seller_state_grouped_SC,main_seller_state_grouped_SP,main_seller_state_grouped_other,main_payment_type_boleto,main_payment_type_credit_card,main_payment_type_debit_card,main_payment_type_voucher,order_id,is_late
0,3.0,1.0,1.0,134.97,8.49,44.99,1.0,104.19,2.0,18.488449,9.0,3.0,0.0,565.959812,3000.0,1000.0,16.0,16.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,bfbd0f9bdef84302105ad712db648a6c,1
1,1.0,1.0,1.0,29.90,15.56,29.90,1.0,45.46,1.0,23.593866,10.0,0.0,0.0,708.754247,300.0,300.0,16.0,16.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3b697a20d9e427646d92567910af6d57,0
2,1.0,1.0,1.0,21.90,17.19,21.90,1.0,39.09,1.0,34.293866,10.0,0.0,0.0,915.912412,400.0,400.0,16.0,16.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,be5bc2f0da14d8071e2d45451ad119d9,0
3,1.0,1.0,1.0,21.50,14.11,21.50,1.0,35.61,1.0,52.123831,10.0,0.0,0.0,357.574592,476.0,476.0,17.0,14.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,65d1e226dfaeb8cdc42f665422522d14,0
4,1.0,1.0,1.0,36.49,17.24,36.49,1.0,53.73,1.0,56.115556,10.0,0.0,0.0,818.146504,767.0,767.0,26.0,8.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,a41c8759fbe7aab36ea07e038b2d4465,0


order_id is kept alongside the features for traceability (so a prediction
can be linked back to a real order later), but it obviously isn't going into
the model as an input, that gets excluded explicitly at training time in
notebook 6.

## Saving everything


In [33]:
train_features.to_parquet(ARTIFACTS_DIR / "train_features.parquet", index=False)
val_features.to_parquet(ARTIFACTS_DIR / "val_features.parquet", index=False)
test_features.to_parquet(ARTIFACTS_DIR / "test_features.parquet", index=False)

joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor.pkl")

with open(ARTIFACTS_DIR / "feature_list.txt", "w") as f:
    f.write("\n".join(feature_names))

print("saved feature tables, preprocessor.pkl, top_states.pkl, top_seller_states.pkl, and feature_list.txt")

saved feature tables, preprocessor.pkl, top_states.pkl, top_seller_states.pkl, and feature_list.txt


Dropped everything that only exists after delivery (delivered dates, review
data, delivery_gap_days), built estimated_delivery_days and a couple of date
features from what's actually known at purchase time, bucketed the long tail
of customer states and seller states using rules learned from train only,
added distance and product weight/dimensions (median imputed where the zip
lookup came up empty), then fit one imputer/encoder pipeline on train and
reused it as-is on val and test.

Saved the three feature tables plus the fitted preprocessor, both state
bucketing lists, and the final feature name list, so notebook 6 (and later,
the production pipeline) never has to refit anything, just load and
transform.

Next: notebook 6, baseline, then a real model, tuned on val, tested once on
test at the very end.